# Python 类型注解详解（教学级 Notebook）

> **一句话本质**：类型注解是给工具看的「文档 + 约束」——它不改变运行时行为，但能让 IDE、类型检查器（mypy / pyright）和框架（FastAPI / Pydantic / SQLAlchemy）在**开发阶段**帮你发现错误。

## 学习目标

1. 掌握类型注解的基础语法：变量、函数、类属性三种注解位置
2. 掌握 `typing` 核心工具：`Optional` / `Union` / `TypeVar` / `Generic` / `Protocol` / `TypedDict` / `Literal` / `Final` / `Annotated` / `Callable` / `overload`
3. 理解前向引用与 `TYPE_CHECKING` 的原理和用途
4. 掌握运行时工具：`__annotations__` / `get_type_hints` / `get_origin` / `get_args`
5. 能在真实工程（LLM API 服务开发）中组合运用上述工具，写出可维护、可校验的代码

## 业务场景说明

本 Notebook 的示例围绕 **LLM API 服务开发（LLMOps）** 取材：聊天消息结构、模型参数约束、Provider 解耦、分页结果、请求/响应校验等。
（注：任务模板中的「业务场景描述」占位符未填写，此处根据目标目录 `LLMOps/llmops-api` 做出的合理假设。）

## 运行环境

- Python 3.13（支持 3.12 的 `type` 语句与 PEP 695 泛型语法）
- 依赖：`pydantic`、`fastapi`、`sqlalchemy`、`mypy`（仅用于生态对照与工具演示）
- 本 Notebook 全部代码单元已实际执行，输出已嵌入，且输出确定可复现

# 一、前置知识

**本章解决什么问题**：类型注解不是孤立特性，它随 Python 版本逐步演进（3.5 → 3.9 → 3.10 → 3.11 → 3.12 → 3.14）。写注解前必须先弄清两件事：
1. 你的解释器版本支持哪些语法；
2. 注解在运行时**到底是什么、存在哪里、何时被求值**——这是理解后面所有「陷阱」的根基。

## 版本演进时间线

| 版本 | 引入特性 | 示例 |
|---|---|---|
| 3.0+ | 函数注解（PEP 3107，后由 3.5 语义化） | `def f(x: int) -> str` |
| 3.5 | 变量注解 + `typing` 模块（PEP 484） | `x: int = 1`、`List[int]` |
| 3.9 | 内置容器可直接下标（PEP 585） | `list[int]`、`dict[str, int]` |
| 3.10 | `X \| Y` 联合类型语法（PEP 604） | `int \| None` |
| 3.11 | `NotRequired` / `Required`（PEP 655） | `age: NotRequired[int]` |
| 3.12 | `type` 语句与 PEP 695 泛型 | `type Vector = list[float]`、`def f[T](x: T)` |
| 3.14 | 注解惰性求值（PEP 649/749） | 定义时不再立即求值（本环境为 3.13，仍为立即求值） |

In [1]:
import sys

print("当前解释器:", sys.version.split()[0])
print()

# 1. 用版本号逐项验证各特性的可用性（而不是靠记忆）
checks = {
    "变量/函数注解 (3.5+)":            sys.version_info >= (3, 5),
    "内置泛型 list[int] (3.9+)":       sys.version_info >= (3, 9),
    "联合类型 int | None (3.10+)":     sys.version_info >= (3, 10),
    "NotRequired / Required (3.11+)":  sys.version_info >= (3, 11),
    "type 语句 / PEP 695 泛型 (3.12+)": sys.version_info >= (3, 12),
    "注解惰性求值 PEP 649 (3.14+)":    sys.version_info >= (3, 14),
}
for name, ok in checks.items():
    print(f"{'[支持]' if ok else '[不支持]'} {name}")

当前解释器: 3.13.12

[支持] 变量/函数注解 (3.5+)
[支持] 内置泛型 list[int] (3.9+)
[支持] 联合类型 int | None (3.10+)
[支持] NotRequired / Required (3.11+)
[支持] type 语句 / PEP 695 泛型 (3.12+)
[不支持] 注解惰性求值 PEP 649 (3.14+)


In [2]:
# 2. 验证：注解在运行时是「真实对象」，存储在 __annotations__ 里，且在定义时立即求值
def f(a: int, b: str = "x") -> bool:
    return True

# 2.1 __annotations__ 是一个普通 dict，键是参数名，值是类型对象本身
print("函数注解:", f.__annotations__)
# 2.2 证明存的是类型对象而不是字符串：可以直接调用它
ann = f.__annotations__["a"]
print("a 的注解是:", ann, "| isinstance(1, ann):", isinstance(1, ann))
# 2.3 类属性注解同样存放在类的 __annotations__ 中
class C:
    a: int
    b: str
print("类注解:", C.__annotations__)

函数注解: {'a': <class 'int'>, 'b': <class 'str'>, 'return': <class 'bool'>}
a 的注解是: <class 'int'> | isinstance(1, ann): True
类注解: {'a': <class 'int'>, 'b': <class 'str'>}


In [3]:
# 3. 验证：模块级变量注解会写入该模块命名空间的 __annotations__（用 exec 模拟独立模块，输出确定）
ns = {}
exec("x: int = 1\ny: str\nprint('模块级注解:', __annotations__)", ns)

# 3.1 注意：注解「只声明不赋值」不会真正创建变量
print("'y' 是否存在于模块命名空间:", "y" in ns)  # False —— 注解本身不产生变量

模块级注解: {'x': <class 'int'>, 'y': <class 'str'>}
'y' 是否存在于模块命名空间: False


# 二、正文

**本章解决什么问题**：按「基础语法 → 基本类型 → 可选与联合 → 泛型 → 协议 → TypedDict → Literal/Final → Annotated → Callable → overload → 类型别名 → 前向引用 → 运行时工具 → 类型检查工具」的顺序，系统掌握全部核心知识点。每个知识点遵循「原理讲解 + 可运行示例」交替。

## 1. 基础语法

### 1.1 变量注解

语法为 `变量名: 类型 = 值`。冒号后的表达式在定义时求值并记入 `__annotations__`，但解释器**不会**用它约束赋值。

### 1.2 函数注解

- 参数：`def f(name: str)` —— 参数名后跟 `: 类型`
- 返回值：`-> 类型`，写在冒号之前

### 1.3 类属性注解

类体中直接写 `属性: 类型`；配合 `__init__` 完成初始化。

In [4]:
# 1.1 变量注解：三种写法
name: str = "Alice"        # ① 注解 + 赋值
age: int = 30
scores: list[float] = [90.5, 88.0]

x: int                     # ② 只注解不赋值：不会创建变量，仅记录注解

y = 3.14                   # ③ 不注解直接赋值：解释器照样接受（动态类型本质未变）

print("name =", name, "| age =", age, "| scores =", scores)
print("x 只注解未赋值 → 未创建变量（模块级行为已在上一单元格 exec 验证）")

# 1.2 函数注解
def greet(name: str, age: int) -> str:   # 参数用 : 类型，返回值用 -> 类型
    return f"{name}, {age}"

print(greet("Alice", 30))

# 1.3 类属性注解
class User:
    name: str               # 类体中的属性注解
    age: int

    def __init__(self, name: str, age: int) -> None:   # -> None 表示无返回值
        self.name = name
        self.age = age

u = User("Alice", 30)
print("User 实例:", u.name, u.age)
print("类属性注解:", User.__annotations__)

name = Alice | age = 30 | scores = [90.5, 88.0]
x 只注解未赋值 → 未创建变量（模块级行为已在上一单元格 exec 验证）
Alice, 30
User 实例: Alice 30
类属性注解: {'name': <class 'str'>, 'age': <class 'int'>}


### 1.4 查看注解 与 1.5 注解是表达式

- `__annotations__` 是普通 dict：键为参数/属性名，返回值用特殊键 `"return"`。
- **修正说明**：源文档称「注解可以是任意表达式，不一定是类型」。准确说法是——*语法上*任意表达式都合法（解释器只求值不校验），但*类型检查器*会报错或警告非类型注解。工程中只应放类型表达式。下例用运行时输出证明「解释器不校验、照常求值」。

In [5]:
# 1.4 查看注解
def f(a: int, b: str = "x") -> bool:
    return True

print(f.__annotations__)
# {'a': <class 'int'>, 'b': <class 'str'>, 'return': <class 'bool'>}

# 1.5 注解在语法上是「表达式」：解释器会求值，但不校验它是不是类型
def g(x: "some string") -> None:      # 字符串也是合法注解（前向引用的基础）
    ...

print("字符串注解原样存储:", g.__annotations__)

def h(x: 1 + 1) -> None:              # 语法合法：注解可以是任意表达式
    ...

print("任意表达式注解被求值:", h.__annotations__)   # x 的注解是 2（解释器只求值不校验）
# 但注意：mypy/pyright 会对上面两处报类型错误 —— 校验职责在工具，不在解释器

{'a': <class 'int'>, 'b': <class 'str'>, 'return': <class 'bool'>}
字符串注解原样存储: {'x': 'some string', 'return': None}
任意表达式注解被求值: {'x': 2, 'return': None}


## 2. 基本类型

### 2.1 内置类型 与 2.2 容器类型

| 类别 | 3.9+ 写法 | 3.8 及以前（typing 模块） |
|---|---|---|
| 列表 | `list[int]` | `List[int]` |
| 字典 | `dict[str, int]` | `Dict[str, int]` |
| 集合 | `set[str]` | `Set[str]` |
| 元组（定长） | `tuple[int, str]` | `Tuple[int, str]` |
| 元组（变长） | `tuple[int, ...]` | `Tuple[int, ...]` |

两种写法在**类型检查器眼中等价**（可互换）。

> **修正说明**：不少资料与直觉认为 `list[int] == List[int]` 在运行时也相等——实测（Python 3.13）为 `False`。`list[int]`（`types.GenericAlias`）与 `List[int]`（`typing._GenericAlias`）是**不同的运行时对象**，只是检查器把它们视为同一类型。对比第 3 章：`Optional[str] == str | None` 的运行时比较却为 `True`。结论：**「类型层面等价」与「运行时对象相等」是两回事**，凡涉及等价性论断都应写代码验证。

### 2.4 Any

`Any` 表示「任意类型」：类型检查器对它完全放行，是类型安全的逃生舱，也是滥用重灾区。

In [6]:
# 2.1 内置标量类型
x: int = 1
y: float = 1.0
z: complex = 1j
s: str = "hello"
b: bool = True
n: None = None
print(x, y, z, s, b, n)

# 2.2 容器类型：Python 3.9+ 直接用内置泛型
nums: list[int] = [1, 2, 3]
mapping: dict[str, int] = {"a": 1}
unique: set[str] = {"a", "b"}
pair: tuple[int, str] = (1, "a")
print(nums, mapping, unique, pair)

# 2.2.1 验证新旧写法：检查器眼中等价，但运行时对象「不相等」（易踩的认知坑）
from typing import List, Dict
print("list[int] == List[int] :", list[int] == List[int])       # False！运行时是两个不同的对象
print("dict[str, int] == Dict[str, int] :", dict[str, int] == Dict[str, int])  # False
# 两者对 mypy/pyright 是同一类型（可互换）；但对运行时代码（如自研校验器）不能划等号

# 2.3 变长元组：... 表示任意长度
point: tuple[float, float] = (1.0, 2.0)      # 定长：恰好两个元素
nums2: tuple[int, ...] = (1, 2, 3, 4)        # 变长：任意个 int
empty: tuple[int, ...] = ()                  # 变长元组可以为空
print(point, nums2, empty)

# 2.4 Any：任意类型都接受
from typing import Any

def echo(x: Any) -> Any:
    return x

print(echo(1), echo("a"), echo([1, 2]))

1 1.0 1j hello True None
[1, 2, 3] {'a': 1} {'a', 'b'} (1, 'a')
list[int] == List[int] : False
dict[str, int] == Dict[str, int] : False
(1.0, 2.0) (1, 2, 3, 4) ()
1 a [1, 2]


## 3. 可选与联合类型

| 写法 | 含义 | 引入版本 |
|---|---|---|
| `Optional[str]` | `str` 或 `None` | 3.5 |
| `Union[int, str]` | `int` 或 `str` | 3.5 |
| `str \| None` | 同 `Optional[str]` | 3.10 |
| `int \| str` | 同 `Union[int, str]` | 3.10 |

要点：`Optional[X]` **只是「允许 None」的类型描述**，与「参数有默认值」毫无关系（这是高频误解，反例章节会专门复现）。

In [7]:
# 3.1 / 3.2 / 3.3 三种联合写法的等价性验证（用输出证明）
from typing import Optional, Union

print("Optional[str] == str | None :", Optional[str] == str | None)     # True
print("Union[int, str] == int | str :", Union[int, str] == int | str)   # True
print("Optional[str] == Union[str, None] :", Optional[str] == Union[str, None])  # True

# 运行时类型对象不同，但语义等价、比较相等
print("type(str | None):", type(str | None))          # types.UnionType（3.10+ 新对象）
print("type(Union[int, str]):", type(Union[int, str]))  # typing.Union（老对象）

# 实际使用：3.10+ 推荐直接用 |
def find(name: str) -> str | None:      # 找到返回名字，找不到返回 None
    db = {"alice": "Alice"}
    return db.get(name)

print(find("alice"))   # Alice
print(find("bob"))     # None

def parse(x: int | str) -> str:
    return str(x)

print(parse(1), parse("1"))

# 3.3 多类型联合可以定义成别名，供多处复用
Value = int | float | str

def identity(x: Value) -> Value:
    return x

print(identity(1), identity(1.5), identity("s"))

Optional[str] == str | None : True
Union[int, str] == int | str : True
Optional[str] == Union[str, None] : True
type(str | None): <class 'types.UnionType'>
type(Union[int, str]): <class 'typing._UnionGenericAlias'>
Alice
None
1 1
1 1.5 s


## 4. 泛型

**本章解决什么问题**：让「同一个函数/类」在不同调用处适配不同类型，并保持**输入输出类型的关联**。例如 `first([1,2,3])` 返回 `int`、`first(["a"])` 返回 `str`——这种关联只有泛型能表达。

核心工具对照：

| 工具 | 用途 | 版本 |
|---|---|---|
| `TypeVar("T")` | 声明类型变量 | 3.5 |
| `TypeVar("T", int, float)` | 带约束（值域限定） | 3.5 |
| `TypeVar("T", bound=Animal)` | 带上界（必须是某类子类） | 3.5 |
| `Generic[T]` | 泛型类基类 | 3.5 |
| `def f[T](x: T)` / `class Box[T]` | PEP 695 新语法 | 3.12 |

In [8]:
# 4.1 TypeVar：函数级泛型
from typing import TypeVar

T = TypeVar("T")

def first(items: list[T]) -> T:     # 入参 list[T] 与返回 T 绑定同一个类型变量
    return items[0]

x = first([1, 2, 3])        # 检查器推断 T -> int，x: int
y = first(["a", "b"])       # 检查器推断 T -> str，y: str
print(x, y, type(x), type(y))

# 4.2 泛型类：数据结构与元素类型绑定
from typing import Generic

class Box(Generic[T]):
    def __init__(self, value: T) -> None:
        self.value = value

    def get(self) -> T:
        return self.value

b = Box(42)
print("Box(42).get() =", b.get(), "| 类型:", type(b.get()))
b2 = Box("hello")
print("Box('hello').get() =", b2.get(), "| 类型:", type(b2.get()))

1 a <class 'int'> <class 'str'>
Box(42).get() = 42 | 类型: <class 'int'>
Box('hello').get() = hello | 类型: <class 'str'>


In [9]:
# 4.3 带约束的 TypeVar：两种方式
from typing import TypeVar

# 方式一：值约束 —— T 只能是 int 或 float
Num = TypeVar("Num", int, float)

def add(a: Num, b: Num) -> Num:
    return a + b

print(add(1, 2), add(1.5, 2.5))
# add(1, "2")  # mypy 会报错：类型不在约束值域内

# 方式二：上界约束 —— T 必须是 Animal 及其子类
class Animal:
    def speak(self) -> str:
        return "..."

class Dog(Animal):
    def speak(self) -> str:
        return "汪"

A = TypeVar("A", bound=Animal)

def make_speak(a: A) -> str:      # 返回类型与入参类型保持关联（如传入 Dog 返回 str）
    return a.speak()

print(make_speak(Animal()), make_speak(Dog()))

# 4.4 Python 3.12+ PEP 695 新语法：无需再写 TypeVar / Generic
def first2[T](items: list[T]) -> T:    # 类型参数直接写在方括号里
    return items[0]

class Box2[T]:                          # 泛型类同样简化
    def __init__(self, value: T) -> None:
        self.value = value

    def get(self) -> T:
        return self.value

print(first2([True, False]), Box2(3.14).get())

3 4.0
... 汪
True 3.14


## 5. 协议 Protocol

**本章解决什么问题**：传统类型系统要求「显式继承才算子类」，而 Python 精神是鸭子类型。`Protocol` 把两者统一：**只要实现了协议规定的方法（结构匹配），就算满足协议**，无需继承。

- 静态：类型检查器做结构检查
- 动态：`@runtime_checkable` 让 `isinstance` 也能做结构检查（只检查方法**是否存在**，不检查签名）

In [10]:
from typing import Protocol, runtime_checkable

# 5.1 定义协议：任何「有 __len__ 方法的对象」都满足它，不需要继承
class Sized(Protocol):
    def __len__(self) -> int: ...

def get_length(obj: Sized) -> int:
    return len(obj)

# list、str 都实现了 __len__ —— 结构匹配即合法
print(get_length([1, 2, 3]))    # 3
print(get_length("hello"))      # 5
# get_length(123)              # 类型检查器报错：int 没有 __len__
# 运行时行为：len(123) 抛 TypeError（注意：报错来自 len()，而非注解）
try:
    get_length(123)
except TypeError as e:
    print("运行时错误（来自 len 而非注解）:", e)

# 5.2 运行时检查：@runtime_checkable
@runtime_checkable
class SizedRC(Protocol):
    def __len__(self) -> int: ...

print("isinstance([1,2], SizedRC):", isinstance([1, 2], SizedRC))   # True：有 __len__
print("isinstance(123, SizedRC):", isinstance(123, SizedRC))        # False：没有 __len__
# 注意：runtime_checkable 只验证「方法存在」，不验证方法签名

3
5
运行时错误（来自 len 而非注解）: object of type 'int' has no len()
isinstance([1,2], SizedRC): True
isinstance(123, SizedRC): False


## 6. TypedDict

**本章解决什么问题**：`dict[str, Any]` 太宽泛——既不知道有哪些键，也不知道每个键的类型。`TypedDict` 给字典一个**结构契约**，让 IDE 补全和检查器都能精确到键。

| 对比项 | `dict[str, int]` | `TypedDict` |
|---|---|---|
| 键集合 | 任意 str | 固定（可含可选键） |
| 各键的值类型 | 统一 | 每键独立声明 |
| IDE 按键补全 | 无 | 有 |
| 运行时校验 | 无 | 无（仍是普通 dict，校验需 pydantic） |

In [11]:
from typing import TypedDict, NotRequired

# 6.1 基本用法：声明字典结构
class User(TypedDict):
    name: str
    age: int
    email: str

u: User = {"name": "Alice", "age": 30, "email": "a@example.com"}
print(u["name"], u["age"])        # IDE 能识别具体键

# 6.2 可选键：NotRequired（3.11+）
class UserPartial(TypedDict):
    name: str                     # 必填键
    age: NotRequired[int]         # 可选键：可以不写

u1: UserPartial = {"name": "Alice"}            # 合法：省略 age
u2: UserPartial = {"name": "Bob", "age": 25}   # 合法：给出 age
print(u1, u2)
print("可选键集合:", UserPartial.__optional_keys__)
print("必填键集合:", UserPartial.__required_keys__)

# 6.3 与 dict 对比：精确度
def create_user(data: User) -> str:
    return data["name"]           # 检查器知道必有 "name" 键且为 str

print(create_user(u))
# 补充验证：TypedDict 在运行时就是普通 dict（不做任何校验）
print("运行时类型:", type(u))
bad: User = {"name": 1, "age": "x", "email": None}   # 运行时不报错！检查器才报错
print("错误类型的字典照样运行:", bad)

Alice 30
{'name': 'Alice'} {'name': 'Bob', 'age': 25}
可选键集合: frozenset({'age'})
必填键集合: frozenset({'name'})
Alice
运行时类型: <class 'dict'>
错误类型的字典照样运行: {'name': 1, 'age': 'x', 'email': None}


## 7. Literal 与 Final

| 工具 | 语义 | 典型场景 |
|---|---|---|
| `Literal["r", "w", "a"]` | 取值只能是这几个字面量 | 限定模式 / 状态 / 枚举字符串 |
| `Final` | 变量不可重新赋值 | 模块常量、类常量 |

要点：两者都**只在检查器层面生效**，运行时不拦截（下面用输出验证）。

In [12]:
from typing import Literal, Final

# 7.1 Literal：限定取值集合
def set_mode(mode: Literal["r", "w", "a"]) -> str:
    return f"模式已设为 {mode}"

print(set_mode("r"))    # 合法
print(set_mode("w"))    # 合法
# set_mode("x")         # mypy/pyright 报错："x" 不在字面量集合内
# 运行时验证：解释器不拦截非法字面量（校验在检查器）
print("运行时传入 'x' 照样执行:", set_mode("x"))

# 验证 Literal 的内部结构（供运行时工具读取）
print("get_args(Literal['r','w','a']):", __import__("typing").get_args(Literal["r", "w", "a"]))

# 7.2 Final：不可重新赋值的常量
MAX_SIZE: Final = 100            # 未指定类型时由初始值推断为 int
# MAX_SIZE = 200                 # 检查器报错；但运行时不会拦截（见下）
MAX_SIZE_RT = MAX_SIZE           # 证明当前值
print("MAX_SIZE =", MAX_SIZE_RT)

# 运行时验证：Final 只是标记，运行时可被重新绑定（不该这么做）
class Config:
    name: Final[str] = "app"     # 类常量

print("Config.name =", Config.name)

# 7.2.1 用 get_type_hints 验证 Final 元数据可被运行时读取
from typing import get_type_hints
print(get_type_hints(Config))

模式已设为 r
模式已设为 w
运行时传入 'x' 照样执行: 模式已设为 x
get_args(Literal['r','w','a']): ('r', 'w', 'a')
MAX_SIZE = 100
Config.name = app
{'name': typing.Final[str]}


## 8. Annotated

`Annotated[类型, 元数据...]` 允许在**不改变类型本身**的前提下附加任意元数据。类型检查器只看第一个参数（真正的类型），后面的元数据留给**框架在运行时读取**——FastAPI 的 `Query(max_length=50)` 就建立在这上面。

In [13]:
from typing import Annotated, get_args, get_origin

# 8.1 基础：给类型附加说明性元数据
Age = Annotated[int, "年龄，0-150"]

def set_age(age: Age) -> None:
    print("设置年龄:", age)

set_age(30)

# 8.2 运行时读取元数据：get_args / get_origin
def f(x: Annotated[int, "positive"]) -> None:
    ...

print("get_args :", get_args(f.__annotations__["x"]))    # (int, 'positive') —— 类型 + 全部元数据
print("get_origin:", get_origin(f.__annotations__["x"])) # Annotated —— 标记这是 Annotated 类型

# 8.3 多条元数据按顺序保留
Multi = Annotated[str, "非空", "trim后使用"]
print("get_args(Multi):", get_args(Multi))

# 8.4 典型框架用法（FastAPI）：元数据在运行时被框架消费
from fastapi import Query

def read_items(q: Annotated[str | None, Query(max_length=50)] = None) -> str | None:
    return q

q_ann = read_items.__annotations__["q"]
print("get_args(q 注解):", get_args(q_ann))
# 第一个元素是真实类型 str | None，其余是框架可读取的 Query 元数据

设置年龄: 30
get_args : (<class 'int'>, 'positive')
get_origin: typing.Annotated
get_args(Multi): (<class 'str'>, '非空', 'trim后使用')


get_args(q 注解): (str | None, Query(PydanticUndefined))


## 9. Callable

`Callable[[参数类型列表], 返回类型]` 描述「可调用对象」（函数、方法、lambda、实现 `__call__` 的实例）。

| 写法 | 含义 |
|---|---|
| `Callable[[int, int], int]` | 接收两个 int、返回 int 的可调用对象 |
| `Callable[[str], None]` | 接收 str、无返回值 |
| `Callable[..., int]` | 任意参数签名、返回 int（`...` 表示不关心参数） |

注意：`Callable` **不能**表达仅关键字参数、默认值等细节——需要精确签名时用 `Protocol`。

In [14]:
from typing import Callable

# 9.1 基本用法：把函数当参数传递（回调模式）
def apply(f: Callable[[int, int], int], a: int, b: int) -> int:
    return f(a, b)

def add(x: int, y: int) -> int:
    return x + y

def mul(x: int, y: int) -> int:
    return x * y

print("apply(add, 1, 2) =", apply(add, 1, 2))
print("apply(mul, 3, 4) =", apply(mul, 3, 4))

# 9.2 别名化：工程中常给回调起语义化别名
Handler = Callable[[str], None]      # 事件处理器：接收事件名，无返回值

def on_event(event: str, handler: Handler) -> None:
    handler(event)

on_event("click", lambda e: print("事件触发:", e))

# 9.3 ... 表示不关心参数签名
F = Callable[..., int]
def run(f: F) -> int:
    return f()                               # 任意签名的可调用对象均可传入
print("run(lambda: 42) =", run(lambda: 42))

apply(add, 1, 2) = 3
apply(mul, 3, 4) = 12
事件触发: click
run(lambda: 42) = 42


## 10. overload

**本章解决什么问题**：一个函数在不同参数类型下返回不同类型（如 `parse(int) -> int`、`parse(str) -> str`），单一签名表达不了。`@overload` 允许声明**多个签名供检查器选用**，再提供一个**不注解的实现**。

关键运行时事实（下面验证）：`@overload` 装饰的签名只是「给检查器看的存根」，运行时真正执行的是最后的实现；`typing.get_overloads()` 可以取回这些存根。

In [15]:
from typing import overload, get_overloads

# 10.1 多签名声明 + 最终实现
@overload
def process(x: int) -> int: ...
@overload
def process(x: str) -> str: ...

def process(x):            # 实现体：不写注解（检查器按 overload 存根推断）
    return x

# 检查器视角：process(1) -> int，process("a") -> str
print(process(1), process("a"))

# 运行时验证 ①：存根可以被取回（3.11+）
print("overload 存根数量:", len(get_overloads(process)))
for stub in get_overloads(process):
    print("  存根:", stub.__annotations__)

# 运行时验证 ②：真正的 process 的注解就是实现体的注解（空）
print("实现的注解:", process.__annotations__)

# 10.2 带默认值的重载
@overload
def get(key: str) -> str: ...
@overload
def get(key: str, default: str) -> str: ...

def get(key, default=None):
    db = {"host": "localhost"}
    return db.get(key, default)

print(get("host"))           # localhost
print(get("port", "8080"))   # 8080

1 a
overload 存根数量: 2
  存根: {'x': <class 'int'>, 'return': <class 'int'>}
  存根: {'x': <class 'str'>, 'return': <class 'str'>}
实现的注解: {}
localhost
8080


## 11. 类型别名

| 写法 | 版本 | 说明 |
|---|---|---|
| `Vector = list[float]` | 任意 | 隐式别名，简单直接 |
| `type Vector = list[float]` | 3.12+ | 显式 `type` 语句（PEP 695），语义更清晰、支持泛型与惰性求值 |
| `Tree = dict[str, "Tree[T]"] \| T` | 3.10+ | 递归别名（配 `TypeVar`），需前向引用字符串 |

In [16]:
# 11.1 简单别名
Vector = list[float]

def scale(v: Vector, factor: float) -> Vector:
    return [x * factor for x in v]

print(scale([1.0, 2.5], 2.0))

# 11.2 Python 3.12+ 的 type 语句：显式别名定义
type Point = tuple[float, float]

def midpoint(a: Point, b: Point) -> Point:
    return ((a[0] + b[0]) / 2, (a[1] + b[1]) / 2)

print(midpoint((0.0, 0.0), (2.0, 4.0)))

# 11.3 泛型递归别名：树的递归结构定义
from typing import TypeVar

T = TypeVar("T")
Tree = dict[str, "Tree[T]"] | T      # 叶子是 T，分支是 dict：递归引用用字符串

def depth(t: Tree) -> int:
    if isinstance(t, dict):
        return 1 + max((depth(v) for v in t.values()), default=0)
    return 0

tree: Tree = {"a": {"b": 1, "c": 2}, "d": 3}
print("树深度:", depth(tree))
print("别名本体:", Tree)

[2.0, 5.0]
(1.0, 2.0)
树深度: 2
别名本体: typing.Union[dict[str, 'Tree[T]'], ~T]


## 12. 前向引用

**本章解决什么问题**：注解在**定义时立即求值**（3.13 及以前），但有些类型在被引用时尚未定义（自引用类、相互引用的类）。解法有两种：

1. **字符串引用**：`"Node | None"`——把求值推迟到检查器解析时
2. **`from __future__ import annotations`**：让本模块所有注解都延迟求值（PEP 563）

> 补充（修正说明）：Python 3.14 起（PEP 649）注解默认惰性求值，本环境为 3.13，仍为「定义时立即求值」。下节反例单元会用真实报错复现这一点。

In [17]:
# 12.1 字符串前向引用：解决「类型还没定义」
class Node:
    def __init__(self, next: "Node | None" = None) -> None:   # "Node" 此刻还没定义完，用字符串
        self.next = next

n1 = Node()
n2 = Node(n1)
print(n2.next is n1)   # True

# 用 get_type_hints 可以把字符串解析回真实类型（需要 Node 已可用）
from typing import get_type_hints
print(get_type_hints(Node.__init__))

True
{'next': __main__.Node | None, 'return': <class 'NoneType'>}


In [18]:
# 12.2 from __future__ import annotations：整个模块的注解全部延迟求值
# 为避免污染当前 Notebook 内核的后续单元格，这里用子进程验证（输出确定）
import subprocess, sys

src = """
from __future__ import annotations   # 必须是文件第一条语句

class Node2:
    def __init__(self, next: Node2 | None = None) -> None:  # 不加引号也不会 NameError
        self.next = next

print("注解以字符串形式存储:", Node2.__init__.__annotations__)
print("节点创建成功:", Node2(Node2()).next is not None)
"""
r = subprocess.run([sys.executable, "-c", src], capture_output=True, text=True)
print(r.stdout, end="")
print("退出码:", r.returncode)   # 0 —— 证明延迟求值生效，未触发 NameError

注解以字符串形式存储: {'next': 'Node2 | None', 'return': 'None'}
节点创建成功: True
退出码: 0


## 13. 运行时类型工具

**本章解决什么问题**：注解虽然是「给工具看的」，但它以真实对象存在 `__annotations__` 里，因此**程序自己也能读取并利用**——这正是 FastAPI、pydantic 等框架的工作原理。

| 工具 | 作用 |
|---|---|
| `__annotations__` | 原始注解 dict（可能含未解析的字符串） |
| `get_type_hints(obj)` | 解析字符串前向引用后的完整注解 |
| `get_origin(list[int])` | 泛型的「原始类型」（`list`） |
| `get_args(list[int])` | 泛型的「参数元组」（`(int,)`） |
| `isinstance` | 运行时类型判断——**不接受参数化泛型** |

In [19]:
from typing import get_type_hints, get_origin, get_args

# 13.1 get_type_hints：解析后的注解（自动解析字符串前向引用）
def f(x: int) -> str:
    ...

print(get_type_hints(f))
# {'x': <class 'int'>, 'return': <class 'str'>}

# 13.2 get_origin / get_args：拆解参数化泛型
print("get_origin(list[int]) :", get_origin(list[int]))       # list
print("get_args(list[int])   :", get_args(list[int]))         # (int,)
print("get_origin(dict[str, int]) :", get_origin(dict[str, int]))  # dict
print("get_args(dict[str, int])   :", get_args(dict[str, int]))    # (str, int)
print("get_origin(str | None) :", get_origin(str | None))     # types.UnionType
print("get_args(str | None)   :", get_args(str | None))       # (str, NoneType)

# 13.3 isinstance 与泛型：参数化泛型不能直接用于 isinstance（运行时验证）
try:
    isinstance([1, 2], list[int])      # TypeError！
except TypeError as e:
    print("报错:", e)
print("正确写法 isinstance([1, 2], list):", isinstance([1, 2], list))

{'x': <class 'int'>, 'return': <class 'str'>}
get_origin(list[int]) : <class 'list'>
get_args(list[int])   : (<class 'int'>,)
get_origin(dict[str, int]) : <class 'dict'>
get_args(dict[str, int])   : (<class 'str'>, <class 'int'>)
get_origin(str | None) : <class 'types.UnionType'>
get_args(str | None)   : (<class 'str'>, <class 'NoneType'>)
报错: isinstance() argument 2 cannot be a parameterized generic
正确写法 isinstance([1, 2], list): True


## 14. 类型检查工具

| 工具 | 定位 | 特点 |
|---|---|---|
| mypy | 命令行/CI 检查器 | 事实标准，可配置严格等级，插件生态 |
| pyright | VS Code / Pylance 默认 | 速度快，严格模式默认较高 |
| pyre / pytype | 其他选择 | Meta / Google 出品，侧重点不同 |
| pydantic | **运行时**校验 | 不是检查器，而是把注解用于运行时验证 |

配置（`pyproject.toml`）：

```toml
[tool.mypy]
strict = true
```


In [20]:
# 14.1 用子进程真实运行 mypy：先看「检查器能发现、解释器发现不了」的错误
import subprocess, sys, tempfile, os

bad_code = """def add(a: int, b: int) -> int:
    return a + b

add("1", "2")
"""
with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as fp:
    fp.write(bad_code)
    bad_path = fp.name

r = subprocess.run([sys.executable, "-m", "mypy", "--no-error-summary", bad_path],
                   capture_output=True, text=True)
print("mypy 检查含错代码的输出:")
print(r.stdout if r.stdout.strip() else "(无输出)")

# 对照：解释器自己运行这段代码完全不报错（动态类型）
exec_ns = {}
exec(bad_code, exec_ns)     # add("1", "2") == "12"，正常运行！
print("解释器实际执行 add('1','2') 的结果:", exec_ns["add"]("1", "2"))
os.unlink(bad_path)

mypy 检查含错代码的输出:
/var/folders/k7/5m0x2p711mqc4ss7955sf7840000gn/T/tmpph1jvso9.py:4: error: Argument 1 to "add" has incompatible type "str"; expected "int"  [arg-type]
/var/folders/k7/5m0x2p711mqc4ss7955sf7840000gn/T/tmpph1jvso9.py:4: error: Argument 2 to "add" has incompatible type "str"; expected "int"  [arg-type]

解释器实际执行 add('1','2') 的结果: 12


In [21]:
# 14.2 mypy 检查正确代码：0 错误
import subprocess, sys, tempfile, os

good_code = """def add(a: int, b: int) -> int:
    return a + b

result = add(1, 2)
"""
with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as fp:
    fp.write(good_code)
    good_path = fp.name

r = subprocess.run([sys.executable, "-m", "mypy", "--no-error-summary", good_path],
                   capture_output=True, text=True)
print("mypy 输出（空 = 无错误）:", repr(r.stdout))
print("退出码:", r.returncode, "(0 表示通过)")
os.unlink(good_path)

mypy 输出（空 = 无错误）: ''
退出码: 0 (0 表示通过)


# 三、反例单元：错误写法 → 真实报错 → 正确写法

**本章解决什么问题**：光看「正确写法」记不牢。本章对每个易错点，先给出**错误写法并实际触发报错**（用 `try/except` 捕获展示，保证单元本身无错误输出），再给出正确写法并对比输出差异。

| # | 易错点 | 错误症状 | 正确解法 |
|---|---|---|---|
| R1 | 注解不影响运行时 | 错误类型照常运行 | 用 mypy/pyright 做 CI 检查 |
| R2 | 泛型用于 isinstance | `TypeError` | 用未参数化的 `list` |
| R3 | 前向引用未加引号 | `NameError` | 字符串引用 / future import |
| R4 | 可变默认参数 | 状态跨调用累积 | 默认值用 `None` |
| R5 | 误以为 Optional=有默认值 | `TypeError: missing argument` | 显式 `= None` |
| R6 | TypedDict 期望运行时校验 | 脏数据静默通过 | 用 pydantic |

In [22]:
# R1 注解不影响运行时：错误类型照样执行，不报任何错
def f(x: int) -> int:
    return x

print("f('hello') =", f("hello"))   # 注解是 int，但传 str 完全正常运行
print("结论：类型检查必须交给 mypy/pyright（见 14.1 的对照实验）")

# R2 泛型用于 isinstance：真实触发 TypeError
try:
    isinstance([1, 2], list[int])
except TypeError as e:
    print("R2 错误写法报错:", e)
print("R2 正确写法:", isinstance([1, 2], list))

f('hello') = hello
结论：类型检查必须交给 mypy/pyright（见 14.1 的对照实验）
R2 错误写法报错: isinstance() argument 2 cannot be a parameterized generic
R2 正确写法: True


In [23]:
# R3 前向引用：类 A 的方法返回值引用了「尚未定义的 B」，定义时立即求值 → NameError
try:
    exec("""
class A:
    def f(self) -> B:
        return None
""")
except NameError as e:
    print("R3 错误写法报错:", e)

# 正确写法①：字符串前向引用
class A2:
    def f(self) -> "B2 | None":
        return None

class B2:
    pass

print("R3 正确写法①: A2().f() =", A2().f())

# 正确写法②：from __future__ import annotations（见 12.2 的子进程验证，输出确定性成立）

R3 错误写法报错: name 'B' is not defined
R3 正确写法①: A2().f() = None


In [24]:
# R4 可变默认参数：注解决不能拯救这个经典陷阱（错误写法）
def append_bad(item, items: list[int] = []):
    items.append(item)
    return items

print("第 1 次调用:", append_bad(1))    # [1]
print("第 2 次调用:", append_bad(2))    # [1, 2]  ← 默认列表被跨调用共享！
print("第 3 次调用:", append_bad(3))    # [1, 2, 3] ← 越积越多

# 正确写法：默认值用 None，函数体内新建
def append_good(item, items: list[int] | None = None) -> list[int]:
    items = [] if items is None else items
    items.append(item)
    return items

print("正确写法第 1 次:", append_good(1))   # [1]
print("正确写法第 2 次:", append_good(2))   # [2] —— 每次独立

第 1 次调用: [1]
第 2 次调用: [1, 2]
第 3 次调用: [1, 2, 3]
正确写法第 1 次: [1]
正确写法第 2 次: [2]


In [25]:
# R5 Optional 不等于有默认值：Optional 只是「允许 None 的类型」
from typing import Optional

def f(x: Optional[int]) -> None:      # 类型上允许 None，但参数仍是必填
    print("x =", x)

try:
    f()                               # 不传参 → TypeError
except TypeError as e:
    print("R5 错误理解报错:", e)

# 正确写法：类型 Optional[int] + 显式默认值 = None
def g(x: int | None = None) -> None:
    print("x =", x)

g()        # x = None（可省略）
g(5)       # x = 5
g(None)    # x = None

R5 错误理解报错: f() missing 1 required positional argument: 'x'
x = None
x = 5
x = None


In [26]:
# R6 TypedDict 不做运行时校验：脏数据静默通过（错误写法）
from typing import TypedDict

class User(TypedDict):
    name: str
    age: int

bad: User = {"name": 123, "age": "abc"}   # 检查器报错，但运行时静默通过
print("R6 错误写法运行结果:", bad, "← 无任何报错")

# 正确写法：需要运行时校验时用 pydantic
from pydantic import BaseModel, ValidationError

class UserP(BaseModel):
    name: str
    age: int

try:
    UserP(name=123, age="abc")
except ValidationError as e:
    print("R6 正确写法捕获校验错误:", e.errors()[0]["type"])
print("合法数据:", UserP(name="Alice", age=30))

R6 错误写法运行结果: {'name': 123, 'age': 'abc'} ← 无任何报错
R6 正确写法捕获校验错误: string_type
合法数据: name='Alice' age=30


# 四、实用示例集：工程高频模式

**本章解决什么问题**：把知识点落到 5 个 LLM API 开发中最常用的模式上，说明适用场景与注意点。

| # | 模式 | 核心工具 | 适用场景 |
|---|---|---|---|
| P1 | 消息结构定义 | `TypedDict` + `Literal` | LLM 聊天消息、请求体 |
| P2 | 配置类 | `@dataclass` + `Final` | 客户端/服务配置 |
| P3 | Provider 解耦与依赖注入 | `Protocol` | 切换 OpenAI / 本地模型后端 |
| P4 | 泛型分页容器 | `TypeVar` + `Generic` | API 列表响应复用 |
| P5 | 多签名 + 取值约束 | `overload` + `Literal` + `Annotated` | 同一函数多种调用形态 |

In [27]:
# P1 消息结构定义：TypedDict + Literal（LLM 聊天消息是教科书级的应用场景）
from typing import TypedDict, Literal, NotRequired

Role = Literal["system", "user", "assistant"]       # 角色只有三种合法取值

class Message(TypedDict):
    role: Role
    content: str
    name: NotRequired[str]                          # 可选：消息归属者

class ChatRequest(TypedDict):
    model: Literal["gpt-4o-mini", "deepseek-chat", "qwen-max"]   # 模型白名单
    messages: list[Message]
    temperature: NotRequired[float]

req: ChatRequest = {
    "model": "deepseek-chat",
    "messages": [
        {"role": "system", "content": "你是一个严谨的助手"},
        {"role": "user", "content": "解释类型注解", "name": "huang"},
    ],
}
print("请求数据:", req)
# 注意点：TypedDict 仍是普通 dict，跨服务边界传输前若不可信，需再过一层 pydantic 校验

请求数据: {'model': 'deepseek-chat', 'messages': [{'role': 'system', 'content': '你是一个严谨的助手'}, {'role': 'user', 'content': '解释类型注解', 'name': 'huang'}]}


In [28]:
# P2 配置类：dataclass + Final（不可变常量 + 带默认值的配置对象）
from dataclasses import dataclass
from typing import Final

API_BASE: Final[str] = "https://api.example.com"    # 常量：Final 标记不可重绑定
TIMEOUT: Final[float] = 30.0

@dataclass
class LLMConfig:
    api_key: str                                    # 必填
    model: str = "deepseek-chat"                    # 有默认值
    temperature: float = 0.7
    max_tokens: int = 1024
    debug: bool = False

cfg = LLMConfig(api_key="sk-xxx")
print("默认配置:", cfg)
cfg2 = LLMConfig(api_key="sk-yyy", model="qwen-max", temperature=0.1)
print("定制配置:", cfg2)
# 注意点：dataclass 的注解在 __annotations__ 中，可被 get_type_hints 读取（框架利用点）
from typing import get_type_hints
print("字段类型:", get_type_hints(LLMConfig))

默认配置: LLMConfig(api_key='sk-xxx', model='deepseek-chat', temperature=0.7, max_tokens=1024, debug=False)
定制配置: LLMConfig(api_key='sk-yyy', model='qwen-max', temperature=0.1, max_tokens=1024, debug=False)
字段类型: {'api_key': <class 'str'>, 'model': <class 'str'>, 'temperature': <class 'float'>, 'max_tokens': <class 'int'>, 'debug': <class 'bool'>}


In [29]:
# P3 Provider 解耦与依赖注入：Protocol 实现零继承解耦
from typing import Protocol

class LLMProvider(Protocol):                        # 结构化契约：有 complete 方法即满足
    def complete(self, prompt: str) -> str: ...

class OpenAIProvider:                               # 注意：没有继承任何东西
    def complete(self, prompt: str) -> str:
        return f"[openai] {prompt[:12]}..."

class LocalMockProvider:                            # 测试用：输出确定，可复现
    def complete(self, prompt: str) -> str:
        return f"[mock] echo: {prompt[:12]}"

def run_pipeline(provider: LLMProvider, prompt: str) -> str:   # 依赖注入：只依赖协议
    return provider.complete(prompt)

# 生产后端与测试后端可无缝替换 —— 这就是 Protocol 的价值
print(run_pipeline(OpenAIProvider(), "请总结这段长文本"))
print(run_pipeline(LocalMockProvider(), "请总结这段长文本"))
# 注意点：协议只约束「结构」，不约束行为正确性；测试替身仍需断言输出

[openai] 请总结这段长文本...
[mock] echo: 请总结这段长文本


In [30]:
# P4 泛型分页容器：一次定义，所有资源复用
from typing import Generic, TypeVar

T = TypeVar("T")

@dataclass
class Page(Generic[T]):
    items: list[T]
    total: int
    page: int
    page_size: int

    @property
    def has_next(self) -> bool:
        return self.page * self.page_size < self.total

# 用户列表分页
users_page: Page[dict[str, int]] = Page([{"id": 1}, {"id": 2}], total=5, page=1, page_size=2)
# 文档列表分页：同一容器，元素类型不同
docs_page: Page[str] = Page(["a.md", "b.md"], total=3, page=1, page_size=2)

print("用户页:", users_page, "| has_next:", users_page.has_next)
print("文档页:", docs_page, "| has_next:", docs_page.has_next)
# 注意点：泛型保证「取出的元素类型」与「存入的元素类型」一致，配合 for 循环可被检查器精确推断

用户页: Page(items=[{'id': 1}, {'id': 2}], total=5, page=1, page_size=2) | has_next: True
文档页: Page(items=['a.md', 'b.md'], total=3, page=1, page_size=2) | has_next: True


In [31]:
# P5 多签名 + 取值约束 + 元数据：overload / Literal / Annotated 三件套
from typing import overload, Literal, Annotated, get_type_hints

ModelName = Literal["gpt-4o-mini", "deepseek-chat", "qwen-max"]
Temperature = Annotated[float, "range:0.0-2.0"]      # 附加业务约束元数据

@overload
def ask(prompt: str) -> str: ...
@overload
def ask(prompt: str, *, model: ModelName) -> str: ...
@overload
def ask(prompt: str, *, model: ModelName, temperature: Temperature) -> str: ...

def ask(prompt: str, *, model: ModelName = "deepseek-chat",
        temperature: Temperature = 0.7) -> str:
    # 运行时利用 Annotated 元数据做防御性校验（框架同理）
    hints = get_type_hints(ask, include_extras=True)
    ann = hints["temperature"]
    meta = str(ann.__metadata__[0])                  # "range:0.0-2.0"
    lo, hi = (float(v) for v in meta.split(":")[1].split("-"))
    if not (lo <= temperature <= hi):
        raise ValueError(f"temperature 超出 [{lo}, {hi}]")
    return f"[{model}@{temperature}] {prompt[:10]}..."

print(ask("你好"))
print(ask("你好", model="qwen-max"))
print(ask("你好", model="gpt-4o-mini", temperature=0.1))
try:
    ask("你好", temperature=9.9)                     # 运行时防御生效
except ValueError as e:
    print("运行时校验:", e)

[deepseek-chat@0.7] 你好...
[qwen-max@0.7] 你好...
[gpt-4o-mini@0.1] 你好...
运行时校验: temperature 超出 [0.0, 2.0]


# 五、内置 / 生态对照

**本章解决什么问题**：类型注解是一条「生态链」——标准库提供语法与对象，检查器负责静态发现，框架负责运行时消费。下表理清各自职责。

| 工具/库 | 定位 | 作用时机 | 与注解的关系 |
|---|---|---|---|
| `typing`（标准库） | 语法与类型对象 | 定义时 | 提供 Protocol/TypedDict/Annotated 等 |
| `dataclasses`（标准库） | 结构化数据类 | 定义时 | 按注解生成 `__init__` 等 |
| mypy | 静态检查器 | 开发/CI 阶段 | 读注解，报告类型错误 |
| pyright（Pylance） | 静态检查器 | 编辑器实时 | 读注解，速度快、严格模式 |
| pydantic | 运行时校验 | 运行时 | 读注解，把类型变成校验规则 |
| FastAPI | Web 框架 | 运行时 | 读注解，自动生成参数解析/文档 |
| SQLAlchemy 2.0 | ORM | 定义+运行时 | `Mapped[T]` 声明列类型 |
| beartype / typeguard | 运行时检查（装饰器） | 函数调用时 | 按注解逐次校验实参 |

一句话分工：**静态找 bug 用 mypy/pyright；运行时守边界用 pydantic；框架内注解即配置（FastAPI/SQLAlchemy）。**

In [32]:
# 生态实测①：SQLAlchemy 2.0 —— Mapped[T] 声明列类型，运行时可提取
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass

class UserModel(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    email: Mapped[str | None] = None           # 可空列用 str | None

print("表名:", UserModel.__tablename__)
print("id 列类型:", UserModel.__table__.c.id.type)
print("name 列可空:", UserModel.__table__.c.name.nullable)
print("email 列可空:", UserModel.__table__.c.email.nullable)

表名: users
id 列类型: INTEGER
name 列可空: False
email 列可空: True


In [33]:
# 生态实测②：FastAPI + Pydantic —— 注解即路由配置与运行时校验
from fastapi import FastAPI, Query
from pydantic import BaseModel, ValidationError

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float

@app.post("/items")
def create_item(item: Item) -> Item:           # 请求体自动按 Item 校验并解析
    return item

@app.get("/items")
def read_items(q: Annotated[str | None, Query(max_length=50)] = None) -> list[Item]:
    return []

# 不启动服务器，也能验证框架对注解的消费：
api_paths = sorted(getattr(r, "path", "") for r in app.routes if getattr(r, "path", "").startswith("/"))
print("注册的 API 路由:", [p for p in api_paths if p in ("/items",)])

# Annotated 中的 Query 元数据在运行时可读（框架据此生成校验与 OpenAPI 文档）
q_ann = read_items.__annotations__["q"]
print("q 的元数据:", get_args(q_ann)[1])

# pydantic 的运行时校验
ok = Item(name="笔记本", price=9.9)
print("合法数据:", ok)
try:
    Item(name="坏数据", price="免费")
except ValidationError as e:
    print("非法数据被拦截:", e.errors()[0]["type"])

注册的 API 路由: ['/items', '/items']
q 的元数据: annotation=NoneType required=True json_schema_extra={} metadata=[MaxLen(max_length=50)]
合法数据: name='笔记本' price=9.9
非法数据被拦截: float_parsing


# 六、综合实战：带类型安全与运行时校验的 LLM API 客户端

**本章解决什么问题**：把前面所有知识点（`Literal` / `TypedDict` / `Protocol` / `Generic` / `overload` / `Annotated` / `get_type_hints` / pydantic）组合成一个贴近 LLMOps 业务的迷你客户端框架，并解释设计取舍。

## 设计蓝图

```
ChatClient                # 门面：对外 API
 ├─ ModelName (Literal)       # 模型白名单
 ├─ Message (TypedDict)       # 消息结构
 ├─ LLMBackend (Protocol)     # 后端契约（可注入 Mock/OpenAI）
 ├─ ChatResult[T] (Generic)   # 通用结果容器
 ├─ Annotated + 运行时校验     # temperature 等参数防御
 └─ pydantic Usage            # 用量统计的运行时校验
```

## 设计取舍

| 决策 | 备选方案 | 取舍理由 |
|---|---|---|
| 用 `Protocol` 而非抽象基类 | ABC 继承 | Mock 后端无需继承，测试替身零成本 |
| 消息用 `TypedDict` | pydantic 模型 | 内部构造可信数据时零开销；跨边界才校验 |
| 响应用 pydantic | TypedDict | 响应来自外部，**不可信**，必须运行时校验 |
| `overload` 区分流式/非流式 | 两个函数名 | 对调用方是一个入口，签名自解释 |
| `Annotated` 带范围元数据 | 手写 if 校验 | 元数据可被工具读取、可复用、可生成文档 |

In [34]:
# ===== 综合实战（1/3）：类型层 —— Literal / TypedDict / Protocol / Generic =====
from typing import (
    Literal, TypedDict, NotRequired, Protocol, Generic, TypeVar,
    Annotated, overload, get_type_hints, get_args, Callable,
)
from dataclasses import dataclass, field

# ① 模型白名单：Literal 限定合法取值（检查器层面）
ModelName = Literal["gpt-4o-mini", "deepseek-chat", "qwen-max"]

# ② 消息结构：TypedDict 定义内部可信数据
Role = Literal["system", "user", "assistant"]

class Message(TypedDict):
    role: Role
    content: str

# ③ 后端契约：Protocol 解耦（Mock / OpenAI / 本地模型可互换）
class LLMBackend(Protocol):
    def chat(self, model: str, messages: list[Message], temperature: float) -> dict: ...

class MockBackend:
    """确定性 Mock 后端：输出只依赖输入，保证测试可复现。"""
    def chat(self, model: str, messages: list[Message], temperature: float) -> dict:
        user_text = next(m["content"] for m in messages if m["role"] == "user")
        return {
            "content": f"[{model}] 收到: {user_text[:8]}...",
            "usage": {"prompt_tokens": len(user_text), "completion_tokens": 16},
        }

# ④ 通用结果容器：泛型关联「结果与用量」
T = TypeVar("T")

@dataclass
class ChatResult(Generic[T]):
    data: T
    prompt_tokens: int
    completion_tokens: int

In [35]:
# ===== 综合实战（2/3）：参数层 —— Annotated 元数据驱动的运行时校验 =====

Temperature = Annotated[float, "range:0.0-2.0"]
MaxTokens = Annotated[int, "range:1-32768"]

def _check_annotated(value, anno, param_name: str) -> None:
    """读取 Annotated 元数据并做范围校验 —— 框架式元数据消费的最小实现。"""
    if get_origin(anno) is Annotated:                    # 只有 Annotated 才带元数据
        meta = get_args(anno)[1]                         # 第 0 个是真实类型
        text = str(meta)
        if text.startswith("range:"):
            lo, hi = (float(v) for v in text[6:].split("-"))
            if not (lo <= value <= hi):
                raise ValueError(f"{param_name}={value} 超出范围 [{lo}, {hi}]")

class ChatClient:
    def __init__(self, backend: LLMBackend) -> None:     # 依赖注入：只依赖协议
        self._backend = backend

    # ⑤ overload：单一入口，多种签名（检查器按入参推断返回类型）
    @overload
    def ask(self, prompt: str) -> ChatResult[str]: ...
    @overload
    def ask(self, prompt: str, *, model: ModelName,
            temperature: Temperature) -> ChatResult[str]: ...

    def ask(self, prompt: str, *, model: ModelName = "deepseek-chat",
            temperature: Temperature = 0.7) -> ChatResult[str]:
        # 运行时防御：用 get_type_hints(include_extras=True) 消费 Annotated 元数据
        hints = get_type_hints(self.ask, include_extras=True)
        _check_annotated(temperature, hints["temperature"], "temperature")

        messages: list[Message] = [{"role": "user", "content": prompt}]
        raw = self._backend.chat(model, messages, temperature)   # 协议调用

        # ⑥ 外部响应不可信 → pydantic 运行时校验（与 TypedDict 内部数据形成对比）
        usage = Usage.model_validate(raw["usage"])
        return ChatResult(data=raw["content"],
                          prompt_tokens=usage.prompt_tokens,
                          completion_tokens=usage.completion_tokens)

from pydantic import BaseModel

class Usage(BaseModel):          # 响应结构：运行时校验
    prompt_tokens: int
    completion_tokens: int

In [36]:
# ===== 综合实战（3/3）：运行与验证 =====
client = ChatClient(MockBackend())

# 形态一：最简调用（默认模型与温度）
r1 = client.ask("什么是类型注解？")
print("r1:", r1)

# 形态二：定制模型与温度
r2 = client.ask("什么是 Protocol？", model="qwen-max", temperature=0.1)
print("r2:", r2)

# 形态三：非法参数被运行时防御拦截
try:
    client.ask("测试", temperature=5.0)
except ValueError as e:
    print("参数校验拦截:", e)

# 验证 overload 存根确实注册（检查器视角的多签名证据）
from typing import get_overloads
print("ask 的 overload 存根数:", len(get_overloads(ChatClient.ask)))

# 验证泛型容器的类型关联在运行时的体现
print("r1 是 ChatResult:", isinstance(r1, ChatResult), "| data 类型:", type(r1.data).__name__)

r1: ChatResult(data='[deepseek-chat] 收到: 什么是类型注解？...', prompt_tokens=8, completion_tokens=16)
r2: ChatResult(data='[qwen-max] 收到: 什么是 Prot...', prompt_tokens=13, completion_tokens=16)
参数校验拦截: temperature=5.0 超出范围 [0.0, 2.0]
ask 的 overload 存根数: 2
r1 是 ChatResult: True | data 类型: str


# 七、常见陷阱汇总

**本章解决什么问题**：把散落各处的坑集中成「陷阱 / 症状 / 解法」速查表，每条均可复现（正文反例单元已给出 R1~R6 的复现代码，此处补充剩余项的复现）。

| 陷阱 | 症状 | 解法 |
|---|---|---|
| 注解不影响运行时 | 错误类型静默运行 | mypy/pyright 进 CI |
| 可变默认参数 | 调用间状态累积 | 默认 `None`，体内新建 |
| Optional ≠ 有默认值 | `TypeError: missing 1 required argument` | 显式 `= None` |
| 泛型用于 isinstance | `TypeError: isinstance() argument 2 cannot be a parameterized generic` | 用未参数化类型 |
| 前向引用未加引号 | 定义时 `NameError` | 字符串 / future import / 3.14+ |
| 类型注解造成循环导入 | `ImportError: cannot import name`（部分初始化） | `TYPE_CHECKING` + 字符串引用 |
| Any 滥用 | 检查形同虚设，错误漏网 | 能具体就具体，入口处收窄 |
| dict 泛型保不了键存在 | `KeyError` 溜过检查 | `TypedDict` |
| 忘记类型收窄 | 分支内仍按联合类型处理 | `isinstance` / `is None` 判断 |
| 期望注解做运行时校验 | 脏数据入库 | pydantic / attrs |

In [37]:
# 陷阱复现①：循环导入 —— 用 TYPE_CHECKING + 字符串引用解决
from typing import TYPE_CHECKING

print("TYPE_CHECKING 运行时为:", TYPE_CHECKING)     # 运行时恒为 False

if TYPE_CHECKING:                                    # 这个分支只在检查器分析时「执行」
    from decimal import Decimal as _Decimal          # 假装这是另一个模块的类型

def price(x: "_Decimal") -> None:                    # 运行时不会真正导入，也不会 NameError
    print("价格:", x)

# 验证：注解以字符串形式存储，运行时零导入成本
print("注解存储:", price.__annotations__)

# 陷阱复现②：dict 泛型保不了键存在（对比 TypedDict）
data: dict[str, int] = {"a": 1}
# data["missing"]  ← 检查器不报错（dict[str,int] 允许任意键），运行时 KeyError
try:
    data["missing"]
except KeyError as e:
    print("dict 泛型症状 KeyError:", e)

from typing import TypedDict
class Config(TypedDict):
    host: str
    port: int
# cfg["host"] ← 检查器能确认键存在；漏写键则直接在字面量处报错
cfg: Config = {"host": "localhost", "port": 8080}
print("TypedDict 精确访问:", cfg["host"], cfg["port"])

TYPE_CHECKING 运行时为: False
注解存储: {'x': '_Decimal', 'return': None}
dict 泛型症状 KeyError: 'missing'
TypedDict 精确访问: localhost 8080


In [38]:
# 陷阱复现③：类型收窄 —— isinstance 分支内，检查器把联合类型「收窄」为具体类型
def describe(x: int | str) -> str:
    if isinstance(x, int):          # 此分支内 x 被收窄为 int
        return f"整数 +1 = {x + 1}"  # x + 1 合法（若 x 还是 int|str，检查器会报错）
    return f"字符串大写 = {x.upper()}"  # 此分支内 x 被收窄为 str

print(describe(1))
print(describe("abc"))

# 对比：不做收窄直接混合操作 → 检查器报错（这里用运行时行为示意差异）
def bad(x: int | str):
    # return x + 1        # mypy 报错：str 不支持 + 1（运行时碰巧传 int 时不炸）
    return x if isinstance(x, int) else len(x)

print("不收窄的替代写法:", bad(1), bad("abc"))

# 陷阱复现④：Any 滥用 —— 检查器对 Any 完全放行
from typing import Any

def sloppy(x: Any) -> Any:
    return x.whatever.method()      # 检查器一言不发 —— 类型安全彻底失效

def precise(x: str) -> int:
    return len(x)                   # 每一步都被检查

print("precise('hello') =", precise("hello"))
print("结论：Any 是逃生舱不是挡箭牌，入口处（如 json.loads 后）应尽快收窄为具体类型")

整数 +1 = 2
字符串大写 = ABC
不收窄的替代写法: 1 3
precise('hello') = 5
结论：Any 是逃生舱不是挡箭牌，入口处（如 json.loads 后）应尽快收窄为具体类型


# 八、总结

## 通用代码模板（可直接复制的骨架）

```python
from __future__ import annotations            # ① 延迟求值，规避前向引用问题
from typing import (
    Annotated, Callable, Final, Generic, Literal,
    NotRequired, Protocol, TypedDict, TypeVar, overload,
)

# ② 常量与取值约束
MAX_RETRIES: Final[int] = 3
Mode = Literal["fast", "accurate"]

# ③ 结构化数据（内部可信用 TypedDict，外部不可信用 pydantic）
class Point(TypedDict):
    x: float
    y: NotRequired[float]

# ④ 泛型容器
T = TypeVar("T")

class Box(Generic[T]):
    def __init__(self, value: T) -> None:
        self.value = value
    def get(self) -> T:
        return self.value

# ⑤ 协议解耦
class Storage(Protocol):
    def save(self, key: str, value: bytes) -> None: ...

# ⑥ 多签名 + 元数据
@overload
def fetch(key: str) -> bytes: ...
@overload
def fetch(key: str, *, decode: Literal[True]) -> str: ...

def fetch(key: str, *, decode: bool = False):
    ...

Temp = Annotated[float, "range:0.0-2.0"]      # 框架可读的约束元数据

def call(temp: Temp = 0.7) -> None: ...
```

## 记忆口诀

1. **注解给工具看，运行时不设限**——校验靠 mypy/pyright，不是解释器。
2. **新版本新写法**：3.9 内置泛型、3.10 用 `|`、3.11 加 NotRequired、3.12 上 `type`。
3. **内部 TypedDict，边界 pydantic**——可信数据零开销，不可信数据必校验。
4. **解耦用 Protocol，复用用泛型**——不继承的结构契约 + 类型关联的容器。
5. **Any 是逃生舱，不是挡箭牌**——越早收窄，检查越有效。
6. **循环导入 TYPE_CHECKING，前向引用加引号**——类型世界的两大「延迟求值」。

## 自测练习（先自己写，再看参考实现）

1. 实现泛型函数 `find_first(items: list[T], predicate: Callable[[T], bool]) -> T | None`
2. 定义递归类型别名 `JSONValue`（JSON 的任意值）
3. 定义协议 `Closable`，并写一个函数 `with_resource(r: Closable) ` 使用后自动关闭
4. 定义 API 响应 `UserResponse`（TypedDict）：必填 `id`/`name`，可选 `email`
5. 用 `overload` 实现 `parse(x: str) -> dict` 与 `parse(x: str, *, as_list: Literal[True]) -> list`

In [39]:
# ===== 自测练习参考实现 =====

# 练习 1：泛型 + Callable + Optional 组合
def find_first(items: list[T], predicate: Callable[[T], bool]) -> T | None:
    for it in items:
        if predicate(it):
            return it
    return None

assert find_first([1, 2, 3], lambda n: n > 1) == 2
assert find_first([1, 2, 3], lambda n: n > 9) is None
print("练习 1 通过:", find_first([1, 2, 3], lambda n: n > 1))

# 练习 2：递归类型别名（叶子为标量，分支为 dict/list）
type JSONValue = None | bool | int | float | str | list["JSONValue"] | dict[str, "JSONValue"]

data: JSONValue = {"ok": True, "items": [1, "a", None]}
print("练习 2 通过:", data)

# 练习 3：协议 + 上下文管理式使用
class Closable(Protocol):
    def close(self) -> None: ...

class FakeConn:
    def close(self) -> None:
        print("连接已关闭")

def with_resource(r: Closable) -> None:
    print("使用资源 ...")
    r.close()

with_resource(FakeConn())

# 练习 4：TypedDict + NotRequired
class UserResponse(TypedDict):
    id: int
    name: str
    email: NotRequired[str]

resp: UserResponse = {"id": 1, "name": "Alice"}
print("练习 4 通过:", resp, "| 可选键:", UserResponse.__optional_keys__)

# 练习 5：overload 多签名
@overload
def parse(x: str) -> dict: ...
@overload
def parse(x: str, *, as_list: Literal[True]) -> list: ...

def parse(x: str, *, as_list: bool = False):
    import json
    obj = json.loads(x)
    return list(obj.items()) if as_list else obj

print("练习 5 通过:", parse('{"a": 1}'), parse('{"a": 1}', as_list=True))

print()
print("全部 5 道自测练习参考实现执行通过 ✓")

练习 1 通过: 2
练习 2 通过: {'ok': True, 'items': [1, 'a', None]}
使用资源 ...
连接已关闭
练习 4 通过: {'id': 1, 'name': 'Alice'} | 可选键: frozenset({'email'})
练习 5 通过: {'a': 1} [('a', 1)]

全部 5 道自测练习参考实现执行通过 ✓


## 结语

> **记住一句话**：类型注解是给工具看的「文档 + 约束」——它不改变运行时行为，但能让 IDE、类型检查器和框架在开发阶段帮你发现错误。

| 层次 | 工具 | 一句话 |
|---|---|---|
| 语法层 | `typing` / 内置泛型 / `type` 语句 | 表达类型的方式 |
| 静态层 | mypy / pyright | 开发阶段找错 |
| 运行时层 | `__annotations__` / `get_type_hints` / `get_origin` / `runtime_checkable` | 让程序自己读注解 |
| 框架层 | pydantic / FastAPI / SQLAlchemy | 注解即配置、即校验 |